In [42]:
# importation de la bibliothèque pandas
import pandas as pd

In [43]:
# importation du dictionnaire de données
df=pd.read_csv('c:/users/NourdiNE/dataset_ventes.csv')

In [44]:
# Calcul du chriffre d'affaire,nombre de commande et panier moyen par client
Resultat=df.groupby('client').agg(CA_total=('montant','sum'),Nbre_commande=('montant','count'),Panier_moyen=('montant','mean')).reset_index()

In [45]:
# conversion de la colone date_commande en format date
df['date_commande']=pd.to_datetime(df['date_commande'])

In [46]:
# création d'une date de réference pour effectuer les calculs
date_reference=df['date_commande'].max()

In [47]:
# détermination de la dernière date d'achat de chaque client
date_achat=df.groupby('client')['date_commande'].max().reset_index()
date_achat.columns=['client','derniere_date_achat']

In [48]:
# calcul du nombre de jour depuis le dernier achat du client
date_achat['recent_jour']=(date_reference - date_achat['derniere_date_achat']).dt.days

In [49]:
# création d'une colonne R_score pour afficher le score de récence
date_achat['R_score']=pd.qcut(date_achat['recent_jour'],5, labels=[5,4,3,2,1])

In [50]:
# création d'une colonne F_score pour afficher le score de fréquence
frequence=df.groupby('client').size().reset_index(name='frequence')
frequence['F_score']=pd.qcut(frequence['frequence'],5, labels=[1,2,3,4,5])

In [51]:
# création d'une colonne M_score pour afficher le score de dépenses
monetaire=df.groupby('client')['montant'].sum().reset_index(name='monetaire')
monetaire['M_score']=pd.qcut(monetaire['monetaire'],5, labels=[1,2,3,4,5])

In [52]:
# Fusion des DataFrame contenant les scores
rfm=date_achat[['client','R_score']].merge(frequence[['client','F_score']], on='client').merge(monetaire[['client','M_score']])

In [53]:
# Création d'une colonne pour segmenté les client selon leur différent score
def segment_client(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
        return 'VIP'
    elif row['F_score'] >= 3:  
        return 'Fidèle'
    elif row['R_score'] <=2:
        return'A risque'
    else:
        return 'Perdu'
rfm['segment']=rfm.apply(segment_client, axis=1)

In [54]:
# Fusion entre les DataFrame Resultat et rfm
Resultat=Resultat.merge(rfm[['client','R_score','F_score','M_score','segment']], on='client')

In [60]:
# Visualisation du DataFame final après traitement
Resultat

,client,CA_total,Nbre_commande,Panier_moyen,R_score,F_score,M_score,segment
0,C001,590.46,10,59.046000,4,5,5,VIP
1,C002,594.66,12,49.555000,5,5,5,VIP
2,C003,393.26,7,56.180000,4,4,4,VIP
3,C004,283.49,6,47.248333,4,3,3,Fidèle
4,C005,205.92,3,68.640000,2,1,2,A risque
...,...,...,...,...,...,...,...,...
74,C075,276.17,6,46.028333,5,3,3,Fidèle
75,C076,301.91,6,50.318333,1,3,3,Fidèle
76,C077,143.26,2,71.630000,1,1,1,A risque
77,C078,221.78,5,44.356000,1,2,2,A risque


In [61]:
# Exportation de fichier sous format .Xlsx(Excel)
Resultat.to_excel('Rapport_client.xlsx',index=False )